# Project Lab 1: Churn, A/B Toolkit and Demand Forecasting

Notebook for **Chapter 15** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch15.html

Tip: change a number, predict the output, then run the cell again.

## 15.1 How to run a project in this area

## 15.2 Project: Churn analysis and retention decision (#3)

**Goal and why employers care.** Predict which shop customers will stop ordering, then decide whom to contact. Most candidates stop at "AUC = 0.75". Employers want the next step: turning a probability into a rupee decision. This is the exact thread of a real retention team.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

def make_customers(n=20000, seed=7):
    r = np.random.default_rng(seed)
    df = pd.DataFrame({
        "tenure_months": r.integers(1, 61, n),
        "orders_90d": r.poisson(3, n),
        "days_since_last": r.integers(0, 120, n),
        "support_tickets": r.poisson(0.8, n),
        "avg_basket": r.gamma(4, 15, n).round(2)})
    z = (-2.0 - 0.03*df.tenure_months - 0.25*df.orders_90d + 0.02*df.days_since_last
         + 0.30*df.support_tickets + 0.004*df.avg_basket)
    p = 1/(1+np.exp(-z))
    df["churned"] = (r.random(n) < p).astype(int)
    df["value"] = (df.avg_basket*df.orders_90d/3*4).round(2)   # rough yearly margin proxy
    return df
# Step 1: data and baseline
df = make_customers()
print(df.shape, "churn rate", round(df.churned.mean(),3))
X = df.drop(columns=["churned","value"]); y = df.churned
Xtr, Xte, ytr, yte, vtr, vte = train_test_split(X, y, df.value, test_size=0.3, random_state=0, stratify=y)
base = max(yte.mean(), 1-yte.mean())
print("majority-class accuracy", round(base,3))
# Step 2: two models, judged by ranking (AUC) and probability quality (Brier)
lr = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
gb = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
for name, m in [("logreg", lr), ("gboost", gb)]:
    p = m.predict_proba(Xte)[:,1]
    print(name, "AUC", round(roc_auc_score(yte,p),3), "Brier", round(brier_score_loss(yte,p),4))
# Step 3: calibration table, 5 equal-size bins
p = gb.predict_proba(Xte)[:,1]
fr, mp = calibration_curve(yte, p, n_bins=5, strategy="quantile")
print("gb calibration (pred -> actual):", [(round(float(a),3),round(float(b),3)) for a,b in zip(mp,fr)])
# Step 4: cost-based targeting: offer cost 5, saves customer with prob 30%, saved value = vte
cost, save = 5.0, 0.30
pl = lr.predict_proba(Xte)[:,1]
gain = pl*save*vte.values - cost          # expected profit per customer if we send
print("send if gain>0:", (gain>0).sum(), "of", len(gain), "profit", round(gain[gain>0].sum(),0))
print("send to all   :", len(gain), "profit", round(gain.sum(),0))
top = np.argsort(-pl)[:int(0.2*len(pl))]
print("top-20% by prob profit", round(gain[top].sum(),0))

## 15.3 Project: A/B test toolkit (#25)

**Goal and why employers care.** Build a small, tested library that does the four jobs every experimentation team needs: size a test, detect a broken split, reduce noise with CUPED, and stop safely. It shows statistical judgement, which is rarer than modelling skill.

In [ ]:
import numpy as np
from scipy import stats

# Step 1: sample size
def n_per_arm(p1, p2, alpha=0.05, power=0.8):
    za, zb = stats.norm.ppf(1-alpha/2), stats.norm.ppf(power)
    pb = (p1+p2)/2
    return (za*np.sqrt(2*pb*(1-pb)) + zb*np.sqrt(p1*(1-p1)+p2*(1-p2)))**2/(p2-p1)**2
print("n per arm:", round(n_per_arm(.10,.11)))

# Step 2: sample-ratio-mismatch check
def srm_p(n_a, n_b, expected=0.5):
    n = n_a+n_b
    return stats.chisquare([n_a,n_b],[n*expected, n*(1-expected)]).pvalue
print("SRM ok  :", round(srm_p(5030,4970),3))
print("SRM bad :", srm_p(5300,4700))

# Step 3: CUPED on a continuous metric (spend), pre-period spend as covariate
r = np.random.default_rng(1)
n = 4000
pre = r.gamma(4, 10, 2*n)
post = 0.8*pre + r.normal(0, 8, 2*n)
t = np.r_[np.zeros(n), np.ones(n)]
post = post + 1.0*t                                 # true lift = +1.0
def diff_se(y, t):
    a, b = y[t==0], y[t==1]
    return b.mean()-a.mean(), np.sqrt(a.var(ddof=1)/len(a)+b.var(ddof=1)/len(b))
theta = np.cov(post, pre)[0,1]/pre.var(ddof=1)
adj = post - theta*(pre-pre.mean())
d0,s0 = diff_se(post,t); d1,s1 = diff_se(adj,t)
print(f"theta={theta:.3f}")
print(f"raw   lift={d0:.3f} se={s0:.3f}")
print(f"CUPED lift={d1:.3f} se={s1:.3f}  var cut={1-(s1/s0)**2:.1%}  corr^2={np.corrcoef(post,pre)[0,1]**2:.3f}")

# Step 4: peeking inflates false alarms: A/A tests, 5 looks, stop at first p<0.05
def aa_false_positive(looks, per_look=400, sims=2000, alpha=0.05, seed=2):
    g = np.random.default_rng(seed); hits=0
    for _ in range(sims):
        a = g.normal(size=per_look*looks); b = g.normal(size=per_look*looks)
        for k in range(1, looks+1):
            m = k*per_look
            p = stats.ttest_ind(a[:m], b[:m]).pvalue
            if p < alpha: hits += 1; break
    return hits/sims
print("one look  FPR:", aa_false_positive(1))
print("five looks FPR:", aa_false_positive(5))
# Step 5: fix: stricter alpha per look: same alpha per look, calibrated by simulation
print("five looks, alpha=0.016 each FPR:", aa_false_positive(5, alpha=0.016))

## 15.4 Project: Demand forecasting with backtesting and intervals (#26)

**Goal and why employers care.** Forecast daily shop demand and prove the model is better than naive guesses, with an honest range around each number. Stock, staffing and delivery plans depend on forecasts, and a forecast without a baseline is just a story.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import Ridge

# Step 1: data
def make_sales(n_days=730, seed=3):
    r = np.random.default_rng(seed)
    d = pd.date_range("2023-01-01", periods=n_days, freq="D")
    t = np.arange(n_days)
    mu = 100 + 0.05*t + 15*np.sin(2*np.pi*t/365.25) + np.where(d.dayofweek>=5, 20, 0)
    return pd.Series(r.poisson(mu).astype(float), index=d)
y = make_sales()

# Step 2: models
def naive(h, hist): return np.repeat(hist.iloc[-1], h)
def seasonal_naive(h, hist): return np.array([hist.iloc[-7 + (i % 7)] for i in range(h)])
def lag_model(h, hist):
    # recursive model: lag1, lag7, dow
    idx = hist.index; v = hist.values
    rows = [(v[i-1], v[i-7], *np.eye(7)[idx[i].dayofweek]) for i in range(7, len(v))]
    m = Ridge(alpha=1.0).fit(np.array(rows), v[7:])
    out=[]; buf=list(v); day=idx[-1]
    for _ in range(h):
        day += pd.Timedelta(days=1)
        out.append(m.predict([[buf[-1], buf[-7], *np.eye(7)[day.dayofweek]]])[0]); buf.append(out[-1])
    return np.array(out)

# Step 3: rolling-origin backtest, 14-day horizon, origin moves 14 days each fold
H, step, start = 14, 14, 365
res = {k: [] for k in ["naive","seasonal_naive","lag_ridge"]}; actuals=[]; resid=[]
for cut in range(start, len(y)-H+1, step):
    hist, fut = y.iloc[:cut], y.iloc[cut:cut+H].values
    actuals.append(fut)
    for k, f in [("naive",naive),("seasonal_naive",seasonal_naive),("lag_ridge",lag_model)]:
        res[k].append(f(H, hist))
A = np.array(actuals)
print("days", len(y), "mean daily sales", round(y.mean(),1))
print("folds", len(A))
for k in res:
    P = np.array(res[k]); mae = np.abs(A-P).mean()
    print(f"{k:15s} MAE={mae:6.2f}  MAPE={np.mean(np.abs(A-P)/A)*100:5.2f}%")
# Step 4: MASE vs naive-7 in-sample scale
scale = np.abs(y.values[7:]-y.values[:-7]).mean()
P = np.array(res["lag_ridge"]); print("MASE lag_ridge:", round(np.abs(A-P).mean()/scale,3))
# Step 5: empirical 80% prediction interval prediction interval from backtest residuals (first half folds calibrate, second half test)
E = A-P; half = len(E)//2
lo, hi = np.quantile(E[:half], [0.1, 0.9])
cov = np.mean((E[half:]>=lo)&(E[half:]<=hi))
print(f"80% interval offsets [{lo:.1f}, {hi:.1f}] coverage on later folds: {cov:.3f}")
# Step 6: rolling calibration: use residuals from the 8 most recent finished folds
hits=[]
for k in range(8, len(E)):
    lo, hi = np.quantile(E[k-8:k], [0.1, 0.9])
    hits.append(np.mean((E[k]>=lo)&(E[k]<=hi)))
print(f"rolling-8-fold 80% interval coverage: {np.mean(hits):.3f} over {len(hits)} folds")

## 15.5 Worked Debugging Problems and a Self-Marking Rubric

The three projects above tell you what to build. This section shows what usually goes wrong and how to find it. Each problem is a bug that real teams ship. Try to spot the bug before you read the steps. Then use the rubric at the end to mark your own project.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(10)
n, cutoff, window = 6000, 100, 30
rate = rng.gamma(2.0, 0.02, n) + 0.005                      # orders per day
last_before = np.zeros(n); cnt90 = np.zeros(n); last_after = np.zeros(n); churn = np.zeros(n)
for i in range(n):
    times = np.cumsum(rng.exponential(1/rate[i], 60))
    times = times[times < cutoff + window]
    before = times[times <= cutoff]; after = times[times > cutoff]
    lb = before.max() if len(before) else 0.0
    last_before[i] = cutoff - lb
    cnt90[i] = ((before > cutoff - 90)).sum()
    churn[i] = float(len(after) == 0)                       # label: no order in next 30 days
    allt = np.r_[before, after]
    last_after[i] = (cutoff + window) - (allt.max() if len(allt) else 0.0)   # recency measured at pull date (leak)
tr, te = slice(0, 4500), slice(4500, 6000)
def auc(X): 
    m = LogisticRegression(max_iter=1000).fit(X[tr], churn[tr]); return roc_auc_score(churn[te], m.predict_proba(X[te])[:, 1])
honest = np.c_[last_before, cnt90]
leaky = np.c_[last_after, cnt90]
print("churn rate:", round(churn.mean(), 3))
print("honest features (recency and count at cutoff) AUC:", round(auc(honest), 3))
print("leaky feature (recency at the pull date) AUC:     ", round(auc(leaky), 3))
print("correlation of leaky recency with label:", round(np.corrcoef(last_after, churn)[0, 1], 3))

churn rate: 0.336
honest features (recency and count at cutoff) AUC: 0.685
leaky feature (recency at the pull date) AUC:      1.0
correlation of leaky recency with label: 0.796


In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(11)
n = 20000
arm = rng.integers(0, 2, n)                                  # fair coin
spend = rng.gamma(2, 10, n)                                  # NO true effect of the treatment
logged = ~((arm == 1) & (spend < 5))                         # bug: treated users with tiny spend lose their record
nA, nB = (arm == 0).sum(), (arm == 1).sum()
print("assigned:", nA, nB, " SRM p =", round(stats.chisquare([nA, nB]).pvalue, 3))
kA, kB = ((arm == 0) & logged).sum(), ((arm == 1) & logged).sum()
print("analysed:", kA, kB, " SRM p =", format(stats.chisquare([kA, kB]).pvalue, ".2e"))
a, b = spend[(arm == 0) & logged], spend[(arm == 1) & logged]
t = stats.ttest_ind(b, a)
print("fake lift in mean spend:", round(b.mean() - a.mean(), 2), " p =", format(t.pvalue, ".3f"))
a0, b0 = spend[arm == 0], spend[arm == 1]
print("lift using everyone assigned:", round(b0.mean() - a0.mean(), 2))

assigned: 9990 10010  SRM p = 0.888
analysed: 9990 9122  SRM p = 3.42e-10
fake lift in mean spend: 1.57  p = 0.000
lift using everyone assigned: -0.04


In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(14)
n = 300
d = pd.Series(100 + 0.1*np.arange(n) + 10*np.sin(2*np.pi*np.arange(n)/7) + rng.normal(0, 6, n))
train, test = d[:240], d[240:]
# bug: smooth the WHOLE series with a centred window, then forecast from the smoothed history
bad_smooth = d.rolling(3, center=True, min_periods=1).mean()
# one-step-ahead forecast = yesterday's smoothed value; compare MAE
f_bad = bad_smooth.shift(1)[240:]
f_ok = d.rolling(3, min_periods=1).mean().shift(1)[240:]          # trailing window only
f_naive = d.shift(1)[240:]
mae = lambda f: float(np.abs(f - test).mean())
print("naive yesterday    MAE:", round(mae(f_naive), 2))
print("trailing mean      MAE:", round(mae(f_ok), 2))
print("centred mean (leak) MAE:", round(mae(f_bad), 2))

naive yesterday    MAE: 8.0
trailing mean      MAE: 8.26
centred mean (leak) MAE: 5.2


## Chapter recap

## Practice set

**1.** (easy) Using the rule of thumb, how many customers per arm to detect a lift of 2 points on a 10% baseline? **Answer:** `16 × 0.1 × 0.9 / 0.02² = 3,600` per arm.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch15.html